In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

# ---------------------------------------------------------------------
# Stage A: structural & initial cleaning for USA raw panel
# ---------------------------------------------------------------------

# Paths
raw_path  = Path("DATA/raw/USA/USA.parquet")
out_dir   = Path("DATA/processed")
out_path  = out_dir / "USA_stageA.parquet"
out_dir.mkdir(parents=True, exist_ok=True)

# Load
df = pd.read_parquet(raw_path)
df = df.copy()  # avoid chained assignment warnings
df["eom"] = pd.to_datetime(df["eom"], errors="coerce")
    
start_date = pd.Timestamp("1963-01-01")
end_date = pd.Timestamp("2024-12-31")
df = df[(df["eom"] >= start_date) & (df["eom"] <= end_date)].copy()
# Keep only non-missing permno (identify tradable CRSP securities)
df = df[df["permno"].notna()].copy()

# Drop duplicates on permno-eom (guard against stale merges)
df = df.drop_duplicates(subset=["permno", "eom"])

# Convert key numeric columns to numeric (coerce errors to NaN)
numeric_cols = [
    "oaccruals_at","at_gr1","be_me","ocf_debt","fcf_me","ret_12_7","ebit_sale",
    "dolvol_126d","div12m_me","be_gr1a","ni_me","sales","ami_126d","ret_12_1",
    "at_be","debtlt_gr1a","rmax1_21d","ret_1_0","ret_6_1","market_equity",
    "taccruals_ni","rvol_21d","ni_be","sale_gr1","sale_me","dolvol_var_126d",
    "turnover_var_126d","turnover_126d","cash_at","assets"
]
for col in numeric_cols:
    if col in df:
        df[col] = pd.to_numeric(df[col], errors="coerce")

# Remove obviously bad return codes and extreme returns
for col in ["ret_1_0", "ret_6_1", "ret_12_1", "ret_12_7"]:
    if col in df:
        # Remove placeholder error codes
        df.loc[df[col].isin([999, 9999, -999, -9999]), col] = np.nan
        # Remove returns above 1000% in absolute value (paper removes >300%, adjust as needed)
        df.loc[df[col].abs() > 3, col] = np.nan

# Handle invalid denominators that lead to explosive ratios:
# - require positive market_equity and assets for log(ME) and salecash
# - require cash_at > 0 and sales > 0 for salecash
df = df[df["market_equity"].gt(0)]
df = df[df["assets"].gt(0)]
df = df[df["cash_at"].gt(0)]
df = df[df["sales"].gt(0)]

# Cap extreme accrual-related ratios and profit ratios
df.loc[df["taccruals_ni"].abs() > 1e3, "taccruals_ni"] = np.nan
df.loc[df["ebit_sale"].abs() > 1e3, "ebit_sale"]       = np.nan

# Save the cleaned Stage A panel
df.to_parquet(out_path, index=False)
print(f"Saved Stage A USA to {out_path}")


/var/folders/1z/df3ckyd548j1xwlq0x0vds080000gn/T/ipykernel_1426/4294907781.py:18: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  df["eom"] = pd.to_datetime(df["eom"], errors="coerce")
/var/folders/1z/df3ckyd548j1xwlq0x0vds080000gn/T/ipykernel

Saved Stage A USA to DATA/processed/USA_stageA.parquet
